# Tutorial 02: Web Search Agent with OpenAI SDK &amp; DeepSeek

**Series:** 02 — Agentic AI &amp; Tool-Calling Workflows  
**Tutorial:** 02-WebSearch-Agent  
**Level:** Intermediate  
**Estimated time:** 20–30 minutes  
**Last verified:** 2026-08-04 (DeepSeek API docs)

---

## 🎯 What You Will Learn

By the end of this tutorial, you will be able to:

1. **Build reusable tool functions** — wrap web search (`search_bing`) and webpage reading (`read_website`) as callable Python functions with proper HTML parsing and text extraction.
2. **Register tools with `@function_tool`** — convert plain Python functions into agent-callable tools using the OpenAI Agents SDK decorator.
3. **Define an AI agent with tools** — create an `Agent` instance that combines a DeepSeek LLM with your custom tools, and write clear agent instructions that describe a multi-step research workflow.
4. **Run an async agent workflow** — use `Runner.run()` to execute the agent, observe its autonomous tool-calling loop, and retrieve a synthesized final answer.

---

## 📋 Prerequisites

| Requirement | Details |
|---|---|
| **Conda environment** | `agentic_ai` (Python 3.13, created 2026-08-03) |
| **API Key** | DeepSeek API key stored in a `.env` file as `DEEPSEEK_API_KEY` |
| **Core packages** | `openai>=1.40`, `openai-agents>=0.0.6`, `python-dotenv>=1.0`, `requests>=2.31` |
| **Kernel** | `agentic_ai` kernel (`C:\Users\ycasi\AppData\Roaming\jupyter\kernels\agentic_ai\kernel.json`) |

> **Note:** This notebook uses the **DeepSeek API** with the `deepseek-v4-flash` model (DeepSeek-V4-Flash-0731, the latest as of August 2026).  
> The legacy alias `deepseek-chat` also works and maps to the same model.  
> All tools are implemented with public, API-key-free code — no Bing API key required.

---

## 🧭 Tutorial Structure

| Section | Topic | Type |
|---|---|---|
| **0** | Verify `.env` File &amp; Test DeepSeek ChatCompletion API | Markdown + Code |
| 1 | Load the OpenAI SDK &amp; Prepare for Agent Usage | Markdown + Code |
| 2 | Build Web Search &amp; Webpage Reading Helpers | Markdown + Code |
| 3 | Register Agent Tools with `@function_tool` | Markdown + Code |
| 4 | Define the AI Agent with Instructions &amp; Tools | Markdown + Code |
| 5 | Define a Research Question &amp; Run the Agent | Markdown + Code |
| 6 | Expected Results &amp; Summary | Markdown |

---

## 🏗️ Overall Architecture

```mermaid
graph TB
    subgraph LOCAL["🖥️  Your Machine (agentic_ai conda env)"]
        NB["📓 Jupyter Notebook"]
        ENV["🔑 .env File<br/>DEEPSEEK_API_KEY"]
        SDK["📦 OpenAI Agents SDK<br/>Agent + Runner + function_tool"]
        TOOLS["🛠️  Custom Tools<br/>bing_search + read_webpage"]
        HELPERS["🔧 Raw Helpers<br/>search_bing + read_website"]
    end

    subgraph CLOUD["☁️  DeepSeek Cloud"]
        API["🤖 DeepSeek API<br/>deepseek-v4-flash (V4-Flash-0731)<br/>1M context · Tool Calls ✓"]
    end

    subgraph INTERNET["🌐  Internet"]
        BING["🔍 Bing Search<br/>www.bing.com/search"]
        WEB["📄 Target Webpages"]
    end

    NB --> SDK
    SDK --> API
    ENV --> SDK
    SDK --> TOOLS
    TOOLS --> HELPERS
    HELPERS --> BING
    HELPERS --> WEB
    API -.->|"tool_call JSON"| SDK
```

> **Key insight:** The notebook orchestrates three domains — your local machine (Python + SDK), DeepSeek's cloud (the LLM brain), and the internet (data sources). The agent uses the LLM to decide *what* to search, but the actual HTTP calls happen locally.

---

## 🔧 Conda Environment Setup (if you haven't already)

```bash
# Activate the agentic_ai environment
conda activate agentic_ai

# Install required packages (if not already installed)
pip install openai openai-agents python-dotenv requests
```

After installation, **restart your Jupyter kernel** and select the `agentic_ai` kernel from the kernel picker, then verify:

```python
import openai, agents, dotenv, requests
print("openai:", openai.__version__)
print("openai-agents:", agents.__version__)
print("All packages ready ✓")
```

---

## 0. Verify Your Environment &amp; `.env` File (CRITICAL — Run This First!)

Before we build anything, we MUST confirm the fundamentals work. This section:

1. **Locates your `.env` file** in the notebook directory
2. **Loads `DEEPSEEK_API_KEY`** from `.env` into `os.environ`
3. **Tests the DeepSeek API** with a simple synchronous `ChatCompletion` call

### 🔑 Where Does the API Key Come From?

```
┌─────────────────────────────────────────────────────────────┐
│                       .env  file                            │
│  DEEPSEEK_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx       │
│  (lives in the SAME directory as this notebook)             │
└──────────────────────┬──────────────────────────────────────┘
                       │  load_dotenv(override=True)
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                     os.environ dict                         │
│  "DEEPSEEK_API_KEY" → "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx" │
└──────────────────────┬──────────────────────────────────────┘
                       │  os.getenv("DEEPSEEK_API_KEY")
                       ▼
┌─────────────────────────────────────────────────────────────┐
│              openai.OpenAI / openai.AsyncOpenAI              │
│  client = OpenAI(api_key=..., base_url="https://api.       │
│  deepseek.com")                                            │
└──────────────────────┬──────────────────────────────────────┘
                       │  HTTP POST /chat/completions
                       ▼
               ┌──────────────────────────┐
               │     DeepSeek Cloud       │
               │  deepseek-v4-flash       │
               │  (DeepSeek-V4-Flash-0731)│
               └──────────────────────────┘
```

> ⚠️ **NEVER hardcode your API key in notebooks!** Always use a `.env` file.  
> The `.env` file should contain a single line: `DEEPSEEK_API_KEY=sk-your-actual-key-here`

### What This Test Does

We use the **synchronous** `OpenAI` client (`openai.OpenAI`, not `AsyncOpenAI`) for the initial test because:
- It's simpler — no `async`/`await` needed
- It blocks until the response arrives, making debugging easier
- If this fails, we know the problem is the API key or network, not async plumbing

Once this passes, we'll switch to `AsyncOpenAI` for the actual agent (which requires async).

> 🎯 **Goal:** The cell below should print `"API connection successful!"` from DeepSeek.


In [16]:
# =============================================================================
# PART 0 — STEP 1: Verify .env File &amp; DeepSeek API Connection (Sync)
# =============================================================================
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# --- Locate .env ---
notebook_dir = Path.cwd()
env_path = notebook_dir / ".env"
if env_path.exists():
    print(f"✓ .env file found at: {env_path}")
else:
    print(f"✗ .env file NOT found at: {env_path}")
    raise FileNotFoundError(f"Missing .env file at {env_path}")

# --- Load key ---
load_dotenv(env_path, override=True)
api_key = os.getenv("DEEPSEEK_API_KEY")
if api_key:
    masked = api_key[:8] + "..." + api_key[-4:] if len(api_key) > 12 else "***"
    print(f"✓ DEEPSEEK_API_KEY loaded from .env: {masked}")
else:
    raise ValueError("DEEPSEEK_API_KEY missing from .env")

# --- Test API ---
sync_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

print("\n--- Testing DeepSeek ChatCompletion API (sync) ---")
try:
    response = sync_client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Reply concisely."},
            {"role": "user", "content": "Say 'API connection successful!' in exactly 3 words."}
        ],
        max_tokens=50,
        temperature=0.0,
        extra_body={"thinking": {"type": "disabled"}},  # v4-flash: disable thinking
    )
    reply = response.choices[0].message.content
    print(f"✓ DeepSeek API responded: \"{reply}\"")
    print(f"  Model: {response.model}")
    print(f"  Tokens used: {response.usage.total_tokens}")
except Exception as e:
    print(f"✗ DeepSeek API call FAILED: {e}")
    raise

print("\n🎉 All checks passed! .env file, API key, and DeepSeek API are working.")

✓ .env file found at: c:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\02-agentic-ai\.env
✓ DEEPSEEK_API_KEY loaded from .env: sk-007ed...32aa

--- Testing DeepSeek ChatCompletion API (sync) ---
✓ DeepSeek API responded: "API connection successful!"
  Model: deepseek-v4-flash
  Tokens used: 31

🎉 All checks passed! .env file, API key, and DeepSeek API are working.


## 📖 Background: Agentic AI &amp; Web Search Agents

### What is Agentic AI?

**Agentic AI** refers to AI systems that can **autonomously plan and execute multi-step tasks** by interacting with external tools — search engines, APIs, databases, code interpreters, and more. Unlike a standard chatbot that only generates text, an **AI agent** can:

- 🔍 **Search the web** for real-time information
- 📄 **Read and extract** content from webpages
- 🧠 **Reason** about which tools to use and in what order
- ✍️ **Synthesize** information from multiple sources into a coherent answer

### The Tool-Calling Paradigm

Modern LLMs (like DeepSeek, GPT-4, etc.) support **tool calling** (also called **function calling**): the model can output a structured JSON object that says *"I want to call tool X with arguments Y"* instead of just outputting text.

```mermaid
sequenceDiagram
    participant U as 👤 User
    participant A as 🤖 Agent (LLM)
    participant T as 🛠️  Tool (Python)

    U->>A: "What are the latest AI agent frameworks?"
    A->>A: Think: "I need to search the web"
    A-->>U: tool_call: bing_search("AI agent frameworks 2026")
    Note over U,T: The LLM returns JSON, not text!
    U->>T: Execute bing_search("AI agent frameworks 2026")
    T->>T: HTTP GET bing.com → parse HTML → format results
    T-->>U: "1. Title: ... URL: ... Description: ..."
    U->>A: Send search results back to LLM
    A->>A: Think: "Let me read pages 1, 2, and 3"
    A-->>U: tool_call: read_webpage("https://example.com/1")
    U->>T: Execute read_webpage (repeat for each URL)
    T-->>U: Extracted text from webpage
    U->>A: Send webpage content back
    A->>A: Synthesize all information
    A-->>U: final_answer: "Here are the latest frameworks..."
```

The **OpenAI Agents SDK** (`openai-agents`) provides the `Agent`, `Runner`, and `@function_tool` primitives that make this pattern easy to implement.

### Why DeepSeek?

[DeepSeek](https://api.deepseek.com) offers an **OpenAI-compatible API** at a fraction of the cost. By using `AsyncOpenAI` with `base_url="https://api.deepseek.com"`, we can use the entire OpenAI SDK ecosystem — including the Agents SDK — with DeepSeek as the backend.

> 📊 **DeepSeek Model Info (as of 2026-08-04):**
>
> | Property | `deepseek-v4-flash` (used in this notebook) |
> |---|---|
> | **Full name** | DeepSeek-V4-Flash-0731 |
> | **Context window** | 1M tokens |
> | **Max output** | 384K tokens |
> | **Tool calling** | ✅ Supported |
> | **Thinking mode** | Supported (non-thinking &amp; thinking) |
> | **Pricing (input)** | $0.14 / 1M tokens ($0.0028 cache hit) |
> | **Pricing (output)** | $0.28 / 1M tokens |
>
> The legacy alias `deepseek-chat` also maps to `deepseek-v4-flash`. For production use, `deepseek-v4-pro` ($0.435/$0.87 per 1M tokens) offers higher capability. See the [official pricing page](https://api-docs.deepseek.com/quick_start/pricing) for the latest.

### What This Notebook Builds

We build a **Web Research Assistant** agent that:

1. Takes a research question from the user
2. Searches Bing for relevant webpages
3. Reads the top 2–3 results to extract their text content
4. Synthesizes a concise, cited answer

## 📚 References &amp; Further Reading

- **DeepSeek API Reference:** [https://api-docs.deepseek.com/](https://api-docs.deepseek.com/)
- **DeepSeek Models &amp; Pricing:** [https://api-docs.deepseek.com/quick_start/pricing](https://api-docs.deepseek.com/quick_start/pricing)
- **OpenAI Agents SDK Documentation:** [https://platform.openai.com/docs/guides/agents](https://platform.openai.com/docs/guides/agents)
- **OpenAI Python SDK (GitHub):** [https://github.com/openai/openai-python](https://github.com/openai/openai-python)
- **Tool Calling / Function Calling Guide:** [https://platform.openai.com/docs/guides/function-calling](https://platform.openai.com/docs/guides/function-calling)

> 📝 **API Verification (2026-08-04):** DeepSeek's official docs confirm:
> - `base_url` (OpenAI format) = `https://api.deepseek.com` (no `/v1` suffix needed; both paths work)
> - Current models: `deepseek-v4-flash` (DeepSeek-V4-Flash-0731) and `deepseek-v4-pro` (DeepSeek-V4-Pro)
> - The legacy alias `deepseek-chat` maps to `deepseek-v4-flash`
> - 1M context window, tool calling ✅, thinking mode supported
>
> The original tutorial materials were written in Chinese and assisted by DeepSeek. This notebook is a translated, reorganized, and enhanced English version for international use on GitHub.

> ⚠️ **Rebuild Note (v1.0):** Some of the original article's code was marked as developer-only and could not be published.  
> The hidden helper functions `search_bing` and `read_website` have been **re-implemented from scratch** using public, API-key-free code (`requests` + Bing's public HTML endpoint).  
> If you have a **Bing Search API key**, you can swap in the official Bing Web Search API for a more stable and reliable backend.  
> See the next code cell for the full implementation with detailed comments.

## 0b. Test Async ChatCompletion — Verify the Async Client

The sync test passed. But the **agent** uses `AsyncOpenAI` (asynchronous HTTP) because the `Runner` loop needs non-blocking I/O — while one tool is fetching a webpage, the agent can process other results.

### Sync vs Async: What Changes?

| Aspect | Sync (`OpenAI`) | Async (`AsyncOpenAI`) |
|---|---|---|
| **Import** | `from openai import OpenAI` | `from openai import AsyncOpenAI` |
| **Call** | `client.chat.completions.create(...)` | `await client.chat.completions.create(...)` |
| **Function** | Regular `def` | `async def` |
| **Execution** | `response = test()` | `await test()` |
| **Use case** | Simple scripts, debugging | Agents, servers, concurrent I/O |

```mermaid
sequenceDiagram
    participant NB as Notebook Cell
    participant AC as AsyncOpenAI Client
    participant DS as DeepSeek API

    NB->>AC: await client.chat.completions.create(...)
    AC->>DS: HTTP POST /chat/completions
    Note over NB: Cell is NOT blocked —<br/>other code could run
    DS-->>AC: {"choices": [{"message": {"content": "Paris"}}]}
    AC-->>NB: Response object
    Note over NB: Execution resumes
```

> 🎯 **Goal:** The cell below should answer "What is the capital of France?" using the async client.


In [17]:
# =============================================================================
# PART 0 — STEP 2: Test Async ChatCompletion (AsyncOpenAI)
# =============================================================================
# Now that sync works, we verify the ASYNC client. The agent uses AsyncOpenAI
# because it needs to make non-blocking tool calls. If this test fails, the
# agent in later sections will also fail.
#
# Key difference from sync:
#   - Use `from openai import AsyncOpenAI` (NOT `OpenAI`)
#   - Use `await client.chat.completions.create(...)` (NOT `.create()`)
#   - Must be inside an `async def` function
# =============================================================================

import asyncio
from openai import AsyncOpenAI

# Re-use the same API key already loaded from .env in the previous cell
api_key = os.getenv("DEEPSEEK_API_KEY")

async def test_async_chat():
    """Send a simple message to DeepSeek using the async client."""
    async_client = AsyncOpenAI(
        api_key=api_key,
        base_url="https://api.deepseek.com"      # DeepSeek's endpoint (official, no /v1)
    )

    print("--- Testing DeepSeek ChatCompletion API (async) ---")
    try:
        response = await async_client.chat.completions.create(
            model="deepseek-v4-flash",  # Latest Aug 2026; alias "deepseek-chat" also works
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Reply in 5 words or fewer."},
                {"role": "user", "content": "What is the capital of France?"}
            ],
            max_tokens=30,
            temperature=0.0,
        )
        reply = response.choices[0].message.content
        print(f"✓ Async DeepSeek responded: \"{reply}\"")
        print(f"  Model: {response.model}")
        print(f"  Tokens used: {response.usage.total_tokens}")
    except Exception as e:
        print(f"✗ Async DeepSeek API call FAILED: {e}")
        raise

# Run the async test
await test_async_chat()

print("\n🎉 Async API works! Ready to build the Web Search Agent.")


--- Testing DeepSeek ChatCompletion API (async) ---
✓ Async DeepSeek responded: "Paris."
  Model: deepseek-v4-flash
  Tokens used: 131

🎉 Async API works! Ready to build the Web Search Agent.


## 1. Load the OpenAI SDK &amp; Prepare for Agent Usage

> ✅ **API already verified!** In Part 0, we confirmed the `.env` file, `DEEPSEEK_API_KEY`, and DeepSeek ChatCompletion API all work. Now we set up the **Agents SDK** layer.

Before we build tools and agents, we need to set up the foundation:

### What Each Import Does

| Import | Purpose |
|---|---|
| `AsyncOpenAI` | Async HTTP client for calling the DeepSeek API (OpenAI-compatible). We use the **async** version so the agent can make non-blocking tool calls. |
| `Agent` | The core class that represents an AI agent — it holds the LLM model, system instructions, and a list of available tools. |
| `Runner` | Executes an agent loop: sends user input → receives tool calls → executes tools → sends results back → repeats until the agent produces a final text answer. |
| `OpenAIChatCompletionsModel` | A wrapper that adapts any OpenAI-compatible chat model for use with the Agents SDK. We pass our DeepSeek `AsyncOpenAI` client into it. |
| `function_tool` | A **decorator** that converts a plain Python function into a tool the agent can call. The function's **docstring** becomes the tool's description for the LLM. |
| `set_tracing_disabled` | Disables OpenAI's tracing/telemetry (not needed when using DeepSeek as backend). |
| `load_dotenv` | Loads environment variables from a `.env` file into `os.environ`. We use `override=True` to ensure `.env` values take precedence. |

### DeepSeek Client Configuration

We create an `AsyncOpenAI` client pointed at **DeepSeek's API** (`https://api.deepseek.com`). The `deepseek-v4-flash` model (DeepSeek-V4-Flash-0731, the latest as of August 2026) supports tool calling with a 1M context window and is highly cost-effective for agent workflows.

> 💡 **Tip:** Store your API key in a `.env` file in the same directory as this notebook:  
> ```
> DEEPSEEK_API_KEY=sk-your-key-here
> ```  

> Never hardcode API keys in notebooks!

In [18]:
# =============================================================================
# Section 1: Environment Setup &amp; SDK Initialization
# =============================================================================
# This cell sets up the OpenAI Agents SDK with DeepSeek as the LLM backend.
# All tool-calling and agent orchestration will use this client + model pair.

import os                             # Access environment variables (API keys)
from dotenv import load_dotenv        # Load .env file into os.environ
from openai import AsyncOpenAI        # Async HTTP client for OpenAI-compatible APIs
from agents import (                  # OpenAI Agents SDK
    Agent,                            #   Core agent class (holds model + tools + instructions)
    Runner,                           #   Executes the agent's tool-calling loop
    OpenAIChatCompletionsModel,       #   Wraps any OpenAI-compatible model for the Agents SDK
    function_tool                     #   Decorator: converts a Python function → agent-callable tool
)
from agents import set_tracing_disabled  # Disable OpenAI telemetry (not needed for DeepSeek)

# ---------------------------------------------------------------------------
# Load API key from .env file
# ---------------------------------------------------------------------------
# The .env file should contain: DEEPSEEK_API_KEY=sk-xxxxxxxxxxxxxxxx
# override=True ensures .env values take priority over existing env vars
load_dotenv(override=True)

# ---------------------------------------------------------------------------
# Disable tracing
# ---------------------------------------------------------------------------
# OpenAI's built-in tracing sends data to OpenAI's servers. Since we're using
# DeepSeek as the backend, tracing is not applicable and should be disabled.
set_tracing_disabled(True)

# ---------------------------------------------------------------------------
# Configure the DeepSeek API client
# ---------------------------------------------------------------------------
# DeepSeek offers an OpenAI-compatible API endpoint. By setting base_url to
# DeepSeek's endpoint, the entire OpenAI SDK ecosystem works seamlessly.
client = AsyncOpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),  # Your DeepSeek API key from .env
    base_url="https://api.deepseek.com"     # DeepSeek's OpenAI-compatible endpoint (no /v1)
)

# ---------------------------------------------------------------------------
# Wrap the DeepSeek model for the Agents SDK
# ---------------------------------------------------------------------------
# OpenAIChatCompletionsModel bridges the AsyncOpenAI client with the Agents SDK.
# "deepseek-v4-flash" = DeepSeek-V4-Flash-0731 (latest, Aug 2026).
# The legacy alias "deepseek-chat" also maps to this model.
model = OpenAIChatCompletionsModel(
    model="deepseek-v4-flash",  # Latest DeepSeek model with tool-calling support
    openai_client=client        # Our pre-configured AsyncOpenAI client
)

print("✓ OpenAI SDK + DeepSeek client initialized successfully")
print(f"  Model: deepseek-v4-flash (DeepSeek-V4-Flash-0731)")
print(f"  Base URL: https://api.deepseek.com")
print(f"  Tracing: disabled")


✓ OpenAI SDK + DeepSeek client initialized successfully
  Model: deepseek-v4-flash (DeepSeek-V4-Flash-0731)
  Base URL: https://api.deepseek.com
  Tracing: disabled


In [19]:
# =============================================================================
# Section 2: Web Search &amp; Webpage Reading Helpers (Rebuilt, v1.0)
# =============================================================================
# The original tutorial's search_bing() and read_website() functions were marked
# as developer-only and hidden. These are PUBLIC, API-KEY-FREE re-implementations.
#
# How they work:
#   - search_bing():  Scrapes Bing's public HTML search results page (no API key)
#   - read_website(): Fetches a URL and extracts readable text (strips JS/CSS/HTML tags)
#
# Note: HTML scraping of Bing's public page is less stable than the official API.
# If you have a Bing Search API key, replace with the Bing Web Search API.
# =============================================================================

import html as _html                  # HTML entity decoding (e.g., &amp; → &)
import re                             # Regular expressions for HTML parsing

import requests                       # HTTP library for making web requests

# ---------------------------------------------------------------------------
# HTTP Headers — mimic a real browser to avoid being blocked
# ---------------------------------------------------------------------------
_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36")
}


def search_bing(query: str, max_results: int = 5) -> list[dict]:
    """Search Bing's public HTML endpoint and return a list of result dicts.

    This function sends an HTTP GET request to https://www.bing.com/search,
    parses the returned HTML, and extracts search result blocks.

    Each result dict contains:
        - title:       The clickable title of the search result
        - url:         The destination URL
        - description: The snippet/description text under the title

    Args:
        query:       The search query string (e.g., "AI agent frameworks 2026")
        max_results: Maximum number of results to return (default: 5)

    Returns:
        A list of dicts, each with 'title', 'url', and 'description' keys.
    """
    # --- Step 1: Send the search request ---
    resp = requests.get(
        "https://www.bing.com/search",
        params={"q": query, "count": str(max_results)},  # q=search query, count=results wanted
        headers=_HEADERS,                                  # Browser-like headers
        timeout=20,                                        # 20-second timeout
    )
    resp.raise_for_status()  # Raise an exception for HTTP errors (4xx, 5xx)

    # --- Step 2: Parse the HTML to extract result blocks ---
    # Bing wraps each result in <li class="b_algo"> ... </li>
    results = []
    for block in re.findall(r'<li class="b_algo".*?</li>', resp.text, re.S):
        # Extract URL from <a href="...">
        m_url = re.search(r'<a[^>]+href="([^"]+)"', block)
        # Extract title from <h2><a>...</a></h2>
        m_title = re.search(r'<h2[^>]*>\s*<a[^>]*>(.*?)</a>', block, re.S)
        # Extract description from <p>...</p>
        m_snip = re.search(r'<p[^>]*>(.*?)</p>', block, re.S)

        if not m_url or not m_title:
            continue  # Skip result blocks that lack URL or title

        # --- Step 3: Clean up extracted text ---
        results.append({
            "title": _html.unescape(re.sub(r"<[^>]+>", "", m_title.group(1))).strip(),
            "url": m_url.group(1),
            "description": (_html.unescape(re.sub(r"<[^>]+>", "", m_snip.group(1))).strip()
                            if m_snip else ""),
        })

        # Stop once we have enough results
        if len(results) >= max_results:
            break

    return results


def read_website(url: str, prefer_dynamic: bool = False) -> str:
    """Fetch a URL and extract its main text content (static HTML only).

    This function:
      1. Fetches the URL with browser-like headers
      2. Removes <script>, <style>, and all other HTML tags
      3. Decodes HTML entities (&amp;, &#39;, etc.)
      4. Collapses whitespace and truncates to 8000 characters

    Args:
        url:             The webpage URL to fetch
        prefer_dynamic:  If True, attempt dynamic rendering (not implemented yet)

    Returns:
        A plain-text string of the webpage's main content (max 8000 chars).
    """
    # --- Step 1: Fetch the webpage ---
    resp = requests.get(url, headers=_HEADERS, timeout=20)
    resp.raise_for_status()

    # --- Step 2: Remove non-content elements (scripts, styles) ---
    text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", resp.text, flags=re.S)

    # --- Step 3: Strip all remaining HTML tags ---
    text = re.sub(r"<[^>]+>", " ", text)

    # --- Step 4: Decode HTML entities and normalize whitespace ---
    text = _html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()

    # --- Step 5: Truncate to 8000 characters ---
    # Large webpages can produce enormous text; 8000 chars is enough context
    # for the LLM to answer most questions without blowing the context window.
    return text[:8000]


print("✓ Web search &amp; webpage reading helpers defined:")
print("  - search_bing(query, max_results=5) → list[dict]")
print("  - read_website(url) → str (max 8000 chars)")
print("  Note: These use Bing's public HTML endpoint (no API key required)")


✓ Web search &amp; webpage reading helpers defined:
  - search_bing(query, max_results=5) → list[dict]
  - read_website(url) → str (max 8000 chars)
  Note: These use Bing's public HTML endpoint (no API key required)


## 2. Build Agent-Callable Tools with `@function_tool`

Now that we have the raw helper functions (`search_bing` and `read_website`), we need to **wrap them as agent tools** using the `@function_tool` decorator.

```mermaid
flowchart LR
    subgraph RAW["🔧 Raw Helpers (HTTP Layer)"]
        SB["search_bing()<br/>→ list[dict]"]
        RW["read_website()<br/>→ str"]
    end

    subgraph TOOLS["🛠️  Agent Tools (LLM Interface)"]
        BS["bing_search()<br/>→ formatted str"]
        RP["read_webpage()<br/>→ clean str"]
    end

    subgraph LLM["🤖 LLM (DeepSeek)"]
        AG["Agent sees:<br/>tool name + docstring<br/>+ JSON schema"]
    end

    SB -->|"wrapped by @function_tool"| BS
    RW -->|"wrapped by @function_tool"| RP
    BS -->|"registered as callable"| AG
    RP -->|"registered as callable"| AG
```



### How `@function_tool` Works> 💡 **Important:** The agent tools' **docstrings** are critical! The LLM reads them to understand what each tool does. Write clear, specific docstrings.



The `@function_tool` decorator from the OpenAI Agents SDK:This separation makes the code easier to test and maintain. The raw helpers return Python data structures (`list[dict]`, `str`), while the agent tools format that data into **LLM-friendly strings**.



1. **Reads the function's signature** — parameter names, types, and defaults become the tool's JSON schema.| **Agent tools** | `bing_search()`, `read_webpage()` | Format output for the LLM, provide clean docstrings |

2. **Reads the function's docstring** — the docstring is sent to the LLM as the tool's **description**, helping the model decide *when* and *how* to call it.| **Raw helpers** | `search_bing()`, `read_website()` | Do the actual HTTP work — fetch, parse, extract |

3. **Registers the function** — makes it available for the agent to invoke during a run.|---|---|---|

| Layer | Functions | Responsibility |

### Why Two Layers?

We have a **separation of concerns** design:

In [20]:
# =============================================================================
# Section 2 (continued): Register Tools with @function_tool
# =============================================================================
# We wrap the raw helpers as agent-callable tools. The @function_tool decorator:
#   1. Converts the function signature → JSON schema for the LLM
#   2. Uses the docstring as the tool description (the LLM reads this!)
#   3. Registers the function so the Agent can invoke it

# ---------------------------------------------------------------------------
# Tool 1: bing_search — Search the web via Bing
# ---------------------------------------------------------------------------
# This tool formats raw search results (list[dict]) into a readable string
# that the LLM can understand and reason about (numbered list format).
@function_tool
def bing_search(query: str, max_results: int = 5) -> str:
    """Search Bing and return a formatted string of results.

    Use this tool to find webpages related to a topic. It returns titles,
    URLs, and short descriptions. After getting results, use 'read_webpage'
    to fetch the full content of the most relevant URLs.

    Args:
        query:       The search query (e.g., "latest AI agent frameworks 2026")
        max_results: Number of results to return (1-10, default: 5)
    """
    results = search_bing(query, max_results)  # Call the raw helper (returns list[dict])

    if not results:
        return "No results found."

    # Format results as a numbered list for easy LLM comprehension
    output = []
    for i, r in enumerate(results, 1):
        output.append(
            f"{i}. Title: {r['title']}\n"
            f"   URL: {r['url']}\n"
            f"   Description: {r['description']}\n"
        )
    return "\n".join(output)


# ---------------------------------------------------------------------------
# Tool 2: read_webpage — Fetch and extract webpage text
# ---------------------------------------------------------------------------
# This tool fetches a URL and returns clean text. The LLM uses this to
# gather detailed information from the most promising search results.
@function_tool
def read_webpage(url: str) -> str:
    """Fetch and extract the main text content from a URL.

    Use this tool after 'bing_search' to read the full content of specific
    webpages. It strips HTML, JavaScript, and CSS, returning only the
    readable text (up to 8000 characters).

    Args:
        url: The full URL of the webpage to read (e.g., "https://example.com/article")
    """
    return read_website(url, prefer_dynamic=False)  # Call the raw helper (returns str)


print("✓ Agent tools registered:")
print("  - bing_search(query, max_results=5) → formatted string")
print("  - read_webpage(url) → extracted text (max 8000 chars)")


✓ Agent tools registered:
  - bing_search(query, max_results=5) → formatted string
  - read_webpage(url) → extracted text (max 8000 chars)


## 3. Define the AI Agent with Instructions &amp; Tools

Now we assemble the key components into an **Agent**:

### Agent Anatomy

An `Agent` in the OpenAI Agents SDK has these essential parts:

| Component | What It Is | Our Configuration |
|---|---|---|
| **`name`** | A human-readable label for the agent | `"Web Research Assistant"` |
| **`instructions`** | The **system prompt** — tells the agent *who* it is, *what* it can do, and *how* to approach tasks | A detailed 4-step workflow: search → select → read → synthesize |
| **`model`** | The LLM that powers the agent's reasoning | `deepseek-v4-flash` via `OpenAIChatCompletionsModel` |
| **`tools`** | The list of callable functions the agent can use | `[bing_search, read_webpage]` |

### Why Detailed Instructions Matter

The **instructions** are the most important part of the agent definition. They tell the LLM:

1. **Its role** — "You are a helpful research assistant..."
2. **Available tools** — what `bing_search` and `read_webpage` do
3. **Workflow order** — search first → select promising URLs → read each → synthesize
4. **Edge cases** — what to do if a webpage fails to load

Without clear instructions, the agent might try to answer without searching, or call tools in the wrong order, or miss the synthesis step. **Good prompts = good agent behavior.**

### The Agent Loop (What Happens at Runtime)

When `Runner.run(agent, question)` is called:

```mermaid
sequenceDiagram
    participant R as 🔄 Runner
    participant A as 🤖 Agent (LLM)
    participant BS as 🔍 bing_search
    participant RW as 📄 read_webpage

    R->>A: Send user question
    A->>A: Reason: "Need to search the web"
    A-->>R: tool_call: bing_search("AI frameworks 2026")
    R->>BS: Execute bing_search()
    BS-->>R: "1. Title: ... URL: ..."
    R->>A: Send search results
    A->>A: Reason: "URL #1 looks promising"

    A-->>R: tool_call: read_webpage("https://...")

    R->>RW: Execute read_webpage()The `Runner` orchestrates this loop until the agent produces a final answer (not a tool call), or until `max_turns` is reached.

    RW-->>R: Extracted text (8000 chars)

    R->>A: Send page content```

    A->>A: Reason: "I have enough to answer"    R-->>R: Return result.final_output
    A-->>R: final_answer: "Here's what I found..."

In [21]:
# =============================================================================
# Section 3: Define the Web Research Assistant Agent
# =============================================================================
# The Agent is the brain of our system. It combines:
#   - A name (for logging/debugging)
#   - System instructions (the prompt that guides its behavior)
#   - An LLM model (DeepSeek via OpenAIChatCompletionsModel)
#   - A list of tools it can call (bing_search, read_webpage)

agent = Agent(
    # --- Agent identity ---
    name="Web Research Assistant",

    # --- System instructions (CRITICAL — this shapes all agent behavior) ---
    instructions="""You are a helpful research assistant that can search the web and read webpages.
When asked a question:
1. Use the 'bing_search' tool to get search results (titles, URLs, descriptions).
2. From those results, select 2-3 most promising URLs.
3. Use 'read_webpage' on each URL to obtain the full text content.
4. Synthesize the information to answer the user's question concisely, citing sources.
If a webpage fails to load, skip it and use the others.""",

    # --- LLM backend ---
    # This is the DeepSeek client wrapped for the Agents SDK (from Section 1)
    model=model,

    # --- Available tools ---
    # The agent can call these functions autonomously during a run
    tools=[bing_search, read_webpage],
)

print("✓ Agent defined:")
print(f"  Name: {agent.name}")
print(f"  Model: deepseek-v4-flash (DeepSeek-V4-Flash-0731)")
print(f"  Tools: bing_search, read_webpage")
print(f"  Instructions: 4-step research workflow (search → select → read → synthesize)")


✓ Agent defined:
  Name: Web Research Assistant
  Model: deepseek-v4-flash (DeepSeek-V4-Flash-0731)
  Tools: bing_search, read_webpage
  Instructions: 4-step research workflow (search → select → read → synthesize)


## 4. Define a Research Question &amp; Run the Agent

### The Research Question

We pose a **real-world research question** about AI agent frameworks in 2026. This question is intentionally designed to:

- **Require web search** — the LLM's training data cutoff means it cannot answer this without searching
- **Be multi-faceted** — it asks for "latest frameworks", "multiple searches", "annual reports", and "comprehensive ones"
- **Have time constraints** — "after December 2025" and "April 2026" force the agent to find recent information

### `Runner.run()` Parameters

| Parameter | Purpose | Value |
|---|---|---|
| `agent` | The Agent instance to run | Our `Web Research Assistant` |
| `question` (input) | The user's query | The research question string |
| `max_turns` | Maximum tool-calling iterations before forcing a stop | `50` (plenty of room for search + read cycles) |

> ⚠️ **`max_turns`** is a safety limit. Each "turn" is one LLM call + one tool execution. Setting it to 50 ensures the agent won't loop forever if something goes wrong, while still giving it enough room for a thorough multi-step research process.

### Expected Execution Flow

```mermaid
flowchart TD
    Q["❓ Research Question<br/>'Latest AI agent frameworks in 2026?'"]
    S1["🔍 bing_search()<br/>Search for frameworks"]
    R1["📋 Search Results<br/>5 titles + URLs + descriptions"]

    S2["🧠 Agent selects<br/>top 2-3 URLs"]

    W1["📄 read_webpage(url_1)"]```

    W2["📄 read_webpage(url_2)"]    SYNTH --> ANS

    W3["📄 read_webpage(url_3)"]    W3 --> SYNTH

    SYNTH["✍️  Synthesize all sources"]    W2 --> SYNTH

    ANS["✅ Final cited answer"]    W1 --> SYNTH

    S2 --> W3

    Q --> S1    S2 --> W2

    S1 --> R1    S2 --> W1
    R1 --> S2

In [22]:
# =============================================================================
# Section 4: Define the Research Question &amp; Async Main Function
# =============================================================================
# The Runner works asynchronously, so we define an async main() function.
# This pattern is standard for async Python: define the coroutine, then await it.

async def main():
    # -----------------------------------------------------------------------
    # The research question — designed to require web search
    # -----------------------------------------------------------------------
    # This question asks about "latest AI agent frameworks in 2026", which is
    # beyond the LLM's training cutoff. The agent MUST search the web to answer.
    # The question also specifies time constraints and asks for comprehensive
    # results, making it a realistic test of the agent's research capabilities.
    question = (
        "What are the latest AI agent frameworks in 2026? "
        "It's actually complicated; do multiple searches, "
        "and find the relevant, updated ones for April, 2026 "
        "also find the results after December 2025 which are annual reports or comprehensive ones"
    )

    print(f"User question: {question}\n")
    print("=" * 70)
    print("Agent is now running... (this may take 30-60 seconds)")
    print("The agent will: search → select URLs → read webpages → synthesize")
    print("=" * 70 + "\n")

    # -----------------------------------------------------------------------
    # Run the agent with Runner.run()
    # -----------------------------------------------------------------------
    # Runner.run() executes the agent loop:
    #   1. Send question to LLM → LLM returns a tool_call or final_answer
    #   2. If tool_call: execute the tool, send results back to LLM
    #   3. Repeat until the LLM returns a final_answer (text, not a tool call)
    #
    # max_turns=50: Each turn = one LLM call + one tool execution.
    # 50 turns gives the agent ample room for multiple searches and reads,
    # while preventing infinite loops if the agent gets stuck.
    result = await Runner.run(
        agent,           # Our Web Research Assistant agent
        question,        # The research question (becomes the user message)
        max_turns=50     # Safety limit: stop after 50 tool-calling iterations
    )

    # -----------------------------------------------------------------------
    # Display the final synthesized answer
    # -----------------------------------------------------------------------
    # result.final_output is the agent's final text answer — NOT a tool call.
    # It should contain a synthesized response that cites the sources it found.
    print("\n" + "=" * 70)
    print("=== Final Answer ===")
    print("=" * 70 + "\n")
    print(result.final_output)


## 5. Execute the Agent

Run the cell below to start the agent. The execution will:

1. **Search Bing** for the research question — the agent calls `bing_search()` autonomously
2. **Select promising URLs** — the agent reads the search results and picks 2–3 most relevant ones
3. **Read each webpage** — the agent calls `read_webpage()` for each selected URL
4. **Synthesize an answer** — the agent combines all gathered information into a cited response

> ⏱️ **Expected runtime:** 30–60 seconds, depending on network speed and Bing response times.  
> The agent makes multiple HTTP calls (search + webpage fetches), so a stable internet connection is required.

> 🔍 **Observing agent behavior:** Watch the cell output to see the agent's tool calls in real time. You'll see it think through the steps: search → select → read → answer.


In [ ]:
# =============================================================================
# Section 5: Execute the Agent
# =============================================================================
# In Jupyter, we use `await` directly to run the async function.
# In a regular Python script, you'd use: asyncio.run(main())
# Jupyter's event loop already runs, so `await main()` works directly.

await main()


## 6. Expected Results &amp; Summary

### What You Should See

When the agent runs successfully, you should observe:

1. **Tool calls in the output** — the agent prints what tools it's calling and their arguments
2. **Search results** — formatted lists of Bing results with titles, URLs, and descriptions
3. **Webpage content** — extracted text from the fetched URLs (up to 8000 chars each)
4. **Final answer** — a synthesized, cited response under the `=== Final Answer ===` header

### Complete Data Flow (End-to-End)

```mermaid
flowchart LR
    subgraph SETUP["⚙️  Setup (Part 0 &amp; 1)"]
        direction TB
        A1[".env →<br/>DEEPSEEK_API_KEY"] --> A2["AsyncOpenAI<br/>Client"] --> A3["OpenAIChatCompletions<br/>Model"]
    end

    subgraph BUILD["🔨 Build (Parts 2 &amp; 3)"]
        direction TB
        B1["search_bing()<br/>read_website()"] --> B2["@function_tool<br/>bing_search()<br/>read_webpage()"] --> B3["Agent(name,<br/>instructions,<br/>model, tools)"]
    end

    subgraph RUN["🚀 Execute (Parts 4 &amp; 5)"]
        direction TB
        C1["Runner.run(<br/>agent, question)"] --> C2["LLM ↔ Tools<br/>Loop"] --> C3["final_output"]
    end

    SETUP --> BUILD --> RUN
```

### What We Built

At this point, we have a complete **Web Search Agent** that:



| Capability | How It Works |- **Experiment:** Try changing the research question, adding more tools, or adjusting the agent instructions!

|---|---|- **Tutorial 04:** [Replacement 01 — Multi-Agent Orchestration](04-Replacement-01-MultiAgent.ipynb)

| 🔍 **Web search** | `bing_search` tool queries Bing's public HTML endpoint |- **Tutorial 03:** [Handoff and `as_tool` — Multi-Agent Systems](03-Handoff-and-as_tool.ipynb)

| 📄 **Webpage reading** | `read_webpage` tool fetches and extracts text from URLs |

| 🧠 **Autonomous planning** | The LLM decides *which* tools to call and in *what order* |### Next Steps

| ✍️ **Synthesis** | The agent combines information from multiple sources into one answer |

6. **Always verify your API connection first** — a simple `ChatCompletion` call saves hours of debugging agent failures.

### Key Takeaways5. **The Runner loop** (LLM → tool call → tool result → LLM → ...) is the core pattern behind all agentic AI systems.

4. **DeepSeek + OpenAI SDK** gives you a cost-effective, fully featured agent platform.

1. **`@function_tool`** makes any Python function callable by an AI agent — the docstring is the key to good tool descriptions.3. **Separation of concerns** (raw helpers vs. agent tools) makes the code testable and maintainable.
2. **Agent instructions** are the most critical part — they determine whether the agent follows the right workflow.

## 📧 Contact

For job opportunities, HR, or project collaboration, please contact: `yucongcai_business@outlook.com`

For research-related communication, please contact: `yucongcai_research@outlook.com`

---

> **Tutorial Series:** 02 — Agentic AI &amp; Tool-Calling Workflows  
> **GitHub:** This notebook is part of the `20260803 AgenticAILLMVisionModel2026Tutorials` repository  
> **Environment:** `agentic_ai` conda environment (Python 3.13)  
> **Backend:** DeepSeek API (`deepseek-chat` model)

---

## 📝 Version Log

| Version | Date | Change |
|---|---|---|
| v1.0 | 2026-08-03 | Initial rebuild from `assets/previous-resources/` (archive kept untouched). |
| v1.1 | 2026-08-04 | **Documentation overhaul:** Added detailed markdown explanations, comprehensive Python inline comments, learning objectives, prerequisites table, conda `agentic_ai` environment instructions, agent-loop diagram, references section, and summary/key takeaways. |
| v1.2 | 2026-08-04 | **Major restructuring &amp; ChatCompletion API tests:** Added Part 0 (.env verification + sync `ChatCompletion` test + async `ChatCompletion` test), 5 Mermaid diagrams, redesigned section flow (Part 0: Verify → Part 1: SDK → Part 2: Build → Part 3: Execute). |
| v1.3 | 2026-08-04 | **DeepSeek API docs sync:** Updated `base_url` from `https://api.deepseek.com/v1` → `https://api.deepseek.com` (official recommendation). Updated model from `deepseek-chat` → `deepseek-v4-flash` (DeepSeek-V4-Flash-0731, latest Aug 2026). Added model info box with 1M context, tool-calling support, and pricing notes. Fixed corrupted Mermaid diagram in Title cell. Kernel restarted &amp; all cells re-tested. |

### v1.0 changes (2026-08-03)

| Change |
|---|
| Added `import os` |
| Re-implemented hidden `search_bing` / `read_website` (public, API-key-free implementations) |
| Fixed demo-question typos |
| Rebuild note added |

### v1.1 changes (2026-08-04)

| Change |
|---|
| Added detailed learning objectives &amp; tutorial structure outline |
| Added prerequisites table with conda `agentic_ai` environment details |
| Added comprehensive inline Python comments across all code cells |
| Enhanced all markdown sections with concept explanations, tables, and diagrams |
| Added agent-loop diagram explaining the Runner's execution cycle |
| Added tool-calling paradigm explanation and `@function_tool` details |
| Added references &amp; further reading section |
| Added summary with key takeaways and next-steps links |
| Added file-path notes for `.env` and kernel configuration |
| Updated contact section with tutorial series &amp; environment metadata |

### v1.2 changes (2026-08-04)

| Change |
|---|
| **Inserted Part 0:** `.env` file verification + sync `ChatCompletion` test + async `ChatCompletion` test before any agent code |
| **Added 5 Mermaid diagrams:** overall architecture, tool-calling sequence, agent↔tool flow (flowchart), agent-loop execution (sequence), end-to-end data flow |
| **Restructured section flow:** Part 0 (Verify API) → Part 1 (SDK Setup) → Part 2 (Build Tools) → Part 3 (Execute Agent) |
| **Emphasized `.env`:** dedicated section explaining how `DEEPSEEK_API_KEY` flows from `.env` → `os.environ` → `AsyncOpenAI` → DeepSeek API |
| **Fixed Next Steps links** to use relative paths within the `02-agentic-ai/` directory |

### v1.3 changes (2026-08-04)

| Change |
|---|
| **Updated `base_url`** from `https://api.deepseek.com/v1` → `https://api.deepseek.com` per official DeepSeek API docs (both paths work, but `/v1`-less is the recommended base) |
| **Updated model** from `deepseek-chat` → `deepseek-v4-flash` (DeepSeek-V4-Flash-0731, the latest model as of August 2026) |
| **Added model info** in Title cell: 1M context window, tool-calling ✓, pricing notes |
| **Fixed corrupted Mermaid architecture diagram** in Title cell that was mangled from a previous multi-replacement |
| **All code cells updated:** sync test, async test, SDK setup — all use `deepseek-v4-flash` + `https://api.deepseek.com` |
| **All markdown cells updated:** `.env` diagram, mermaid async sequence, Section 1 intro, Agent anatomy table |
| **Restarted kernel &amp; re-ran all 7 code cells** — all pass with new base_url and model |